In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

sys.path.insert(0, '..')

from src.notebook_utils import load_eb_catalog, load_panstarrs_data, MISSING_VALUE
from src.config import get_config

# Load the Gaia catalog with proper motion
print("Loading Gaia catalog with proper motion...")
gaia_pm = load_eb_catalog(with_pm=True, format='pandas')
print(f"Gaia PM catalog shape: {gaia_pm.shape}")

# Load the Pan-STARRS photometry with temperatures
print("Loading Pan-STARRS photometry with temperatures...")
panstarrs_temp = load_panstarrs_data(format='pandas')
print(f"Pan-STARRS temperature catalog shape: {panstarrs_temp.shape}")

# Join the tables on original_ext_source_id
print("Joining tables on original_ext_source_id...")
merged = gaia_pm.merge(panstarrs_temp, on='original_ext_source_id', how='inner')
print(f"Merged catalog shape: {merged.shape}")

print(f"\nMerged catalog columns: {len(merged.columns)}")
print(f"Available temperature measurements:")
print(f"  Te_avg != {MISSING_VALUE}: {(merged['Te_avg'] != MISSING_VALUE).sum():,} objects")
print(f"  Te_gr != {MISSING_VALUE}: {(merged['Te_gr'] != MISSING_VALUE).sum():,} objects")
print(f"  Te_ri != {MISSING_VALUE}: {(merged['Te_ri'] != MISSING_VALUE).sum():,} objects")
print(f"  Te_iz != {MISSING_VALUE}: {(merged['Te_iz'] != MISSING_VALUE).sum():,} objects")

merged.head()


In [ ]:
# Plot Gaia T_eff vs Pan-STARRS Te_avg
# Filter for valid temperature measurements
valid_temp = (merged['teff_gspphot'] > 0) & (merged['Te_avg'] != MISSING_VALUE)
temp_data = merged[valid_temp]

print(f"Objects with both Gaia T_eff and Pan-STARRS Te_avg: {len(temp_data):,}")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 2D histogram (hexbin plot)
axes[0,0].hexbin(temp_data['teff_gspphot'], temp_data['Te_avg'],
                gridsize=50, cmap='plasma', mincnt=1)
axes[0,0].set_xlabel('Gaia T_eff (K)')
axes[0,0].set_ylabel('Pan-STARRS Te_avg (K)')
axes[0,0].set_title('2D Histogram: Gaia vs Pan-STARRS Temperature')
axes[0,0].plot([3000, 8000], [3000, 8000], 'w--', alpha=0.7, label='1:1 line')
axes[0,0].legend()

# Density plot with contours
axes[0,1].hist2d(temp_data['teff_gspphot'], temp_data['Te_avg'],
                bins=100, cmap='viridis', density=True)
axes[0,1].set_xlabel('Gaia T_eff (K)')
axes[0,1].set_ylabel('Pan-STARRS Te_avg (K)')
axes[0,1].set_title('Density Plot with Contours')
axes[0,1].plot([3000, 8000], [3000, 8000], 'w--', alpha=0.7)

# Residuals vs Gaia T_eff
residuals = temp_data['Te_avg'] - temp_data['teff_gspphot']
axes[1,0].hexbin(temp_data['teff_gspphot'], residuals,
                gridsize=50, cmap='coolwarm', mincnt=1)
axes[1,0].set_xlabel('Gaia T_eff (K)')
axes[1,0].set_ylabel('Te_avg - T_eff (K)')
axes[1,0].set_title('Temperature Residuals')
axes[1,0].axhline(y=0, color='white', linestyle='--', alpha=0.7)

# Distribution of residuals
axes[1,1].hist(residuals, bins=100, alpha=0.7, density=True)
axes[1,1].set_xlabel('Te_avg - T_eff (K)')
axes[1,1].set_ylabel('Density')
axes[1,1].set_title('Residual Distribution')
axes[1,1].axvline(x=0, color='red', linestyle='--', alpha=0.7)
axes[1,1].axvline(x=np.median(residuals), color='orange', linestyle='-',
                    label=f'Median: {np.median(residuals):.0f} K')
axes[1,1].legend()

plt.tight_layout()
plt.show()

# Print statistics
print(f"\nTemperature comparison statistics:")
print(f"Gaia T_eff range: {temp_data['teff_gspphot'].min():.0f} - {temp_data['teff_gspphot'].max():.0f} K")
print(f"Pan-STARRS Te_avg range: {temp_data['Te_avg'].min():.0f} - {temp_data['Te_avg'].max():.0f} K")
print(f"Median residual (Te_avg - T_eff): {np.median(residuals):.0f} K")
print(f"MAD residual: {np.median(np.abs(residuals - np.median(residuals))):.0f} K")
print(f"RMS residual: {np.sqrt(np.mean(residuals**2)):.0f} K")


In [ ]:
# Plot Gaia T_eff vs Pan-STARRS Te_gr
# Filter for valid temperature measurements
valid_temp = (merged['teff_gspphot'] > 0) & (merged['Te_gr'] != MISSING_VALUE)
temp_data = merged[valid_temp]

print(f"Objects with both Gaia T_eff and Pan-STARRS Te_gr: {len(temp_data):,}")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 2D histogram (hexbin plot)
axes[0,0].hexbin(temp_data['teff_gspphot'], temp_data['Te_gr'],
                gridsize=50, cmap='plasma', mincnt=1)
axes[0,0].set_xlabel('Gaia T_eff (K)')
axes[0,0].set_ylabel('Pan-STARRS Te_gr (K)')
axes[0,0].set_title('2D Histogram: Gaia vs Pan-STARRS Temperature')
axes[0,0].plot([3000, 8000], [3000, 8000], 'w--', alpha=0.7, label='1:1 line')
axes[0,0].legend()

# Density plot with contours
axes[0,1].hist2d(temp_data['teff_gspphot'], temp_data['Te_gr'],
                bins=100, cmap='viridis', density=True)
axes[0,1].set_xlabel('Gaia T_eff (K)')
axes[0,1].set_ylabel('Pan-STARRS Te_gr (K)')
axes[0,1].set_title('Density Plot with Contours')
axes[0,1].plot([3000, 8000], [3000, 8000], 'w--', alpha=0.7)

# Residuals vs Gaia T_eff
residuals = temp_data['Te_gr'] - temp_data['teff_gspphot']
axes[1,0].hexbin(temp_data['teff_gspphot'], residuals,
                gridsize=50, cmap='coolwarm', mincnt=1)
axes[1,0].set_xlabel('Gaia T_eff (K)')
axes[1,0].set_ylabel('Te_gr - T_eff (K)')
axes[1,0].set_title('Temperature Residuals')
axes[1,0].axhline(y=0, color='white', linestyle='--', alpha=0.7)

# Distribution of residuals
axes[1,1].hist(residuals, bins=100, alpha=0.7, density=True)
axes[1,1].set_xlabel('Te_gr - T_eff (K)')
axes[1,1].set_ylabel('Density')
axes[1,1].set_title('Residual Distribution')
axes[1,1].axvline(x=0, color='red', linestyle='--', alpha=0.7)
axes[1,1].axvline(x=np.median(residuals), color='orange', linestyle='-',
                    label=f'Median: {np.median(residuals):.0f} K')
axes[1,1].legend()

plt.tight_layout()
plt.show()

# Print statistics
print(f"\nTemperature comparison statistics:")
print(f"Gaia T_eff range: {temp_data['teff_gspphot'].min():.0f} - {temp_data['teff_gspphot'].max():.0f} K")
print(f"Pan-STARRS Te_gr range: {temp_data['Te_gr'].min():.0f} - {temp_data['Te_gr'].max():.0f} K")
print(f"Median residual (Te_gr - T_eff): {np.median(residuals):.0f} K")
print(f"MAD residual: {np.median(np.abs(residuals - np.median(residuals))):.0f} K")
print(f"RMS residual: {np.sqrt(np.mean(residuals**2)):.0f} K")


In [ ]:
# Plot Gaia T_eff vs Pan-STARRS Te_ri
# Filter for valid temperature measurements
valid_temp = (merged['teff_gspphot'] > 0) & (merged['Te_ri'] != MISSING_VALUE)
temp_data = merged[valid_temp]

print(f"Objects with both Gaia T_eff and Pan-STARRS Te_ri: {len(temp_data):,}")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 2D histogram (hexbin plot)
axes[0,0].hexbin(temp_data['teff_gspphot'], temp_data['Te_ri'],
                gridsize=50, cmap='plasma', mincnt=1)
axes[0,0].set_xlabel('Gaia T_eff (K)')
axes[0,0].set_ylabel('Pan-STARRS Te_ri (K)')
axes[0,0].set_title('2D Histogram: Gaia vs Pan-STARRS Temperature')
axes[0,0].plot([3000, 8000], [3000, 8000], 'w--', alpha=0.7, label='1:1 line')
axes[0,0].legend()

# Density plot with contours
axes[0,1].hist2d(temp_data['teff_gspphot'], temp_data['Te_ri'],
                bins=100, cmap='viridis', density=True)
axes[0,1].set_xlabel('Gaia T_eff (K)')
axes[0,1].set_ylabel('Pan-STARRS Te_ri (K)')
axes[0,1].set_title('Density Plot with Contours')
axes[0,1].plot([3000, 8000], [3000, 8000], 'w--', alpha=0.7)

# Residuals vs Gaia T_eff
residuals = temp_data['Te_ri'] - temp_data['teff_gspphot']
axes[1,0].hexbin(temp_data['teff_gspphot'], residuals,
                gridsize=50, cmap='coolwarm', mincnt=1)
axes[1,0].set_xlabel('Gaia T_eff (K)')
axes[1,0].set_ylabel('Te_ri - T_eff (K)')
axes[1,0].set_title('Temperature Residuals')
axes[1,0].axhline(y=0, color='white', linestyle='--', alpha=0.7)

# Distribution of residuals
axes[1,1].hist(residuals, bins=100, alpha=0.7, density=True)
axes[1,1].set_xlabel('Te_ri - T_eff (K)')
axes[1,1].set_ylabel('Density')
axes[1,1].set_title('Residual Distribution')
axes[1,1].axvline(x=0, color='red', linestyle='--', alpha=0.7)
axes[1,1].axvline(x=np.median(residuals), color='orange', linestyle='-',
                    label=f'Median: {np.median(residuals):.0f} K')
axes[1,1].legend()

plt.tight_layout()
plt.show()

# Print statistics
print(f"\nTemperature comparison statistics:")
print(f"Gaia T_eff range: {temp_data['teff_gspphot'].min():.0f} - {temp_data['teff_gspphot'].max():.0f} K")
print(f"Pan-STARRS Te_ri range: {temp_data['Te_ri'].min():.0f} - {temp_data['Te_ri'].max():.0f} K")
print(f"Median residual (Te_ri - T_eff): {np.median(residuals):.0f} K")
print(f"MAD residual: {np.median(np.abs(residuals - np.median(residuals))):.0f} K")
print(f"RMS residual: {np.sqrt(np.mean(residuals**2)):.0f} K")


In [ ]:
# Plot Gaia T_eff vs Pan-STARRS Te_iz
# Filter for valid temperature measurements
valid_temp = (merged['teff_gspphot'] > 0) & (merged['Te_iz'] != MISSING_VALUE)
temp_data = merged[valid_temp]

print(f"Objects with both Gaia T_eff and Pan-STARRS Te_iz: {len(temp_data):,}")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 2D histogram (hexbin plot)
axes[0,0].hexbin(temp_data['teff_gspphot'], temp_data['Te_iz'],
                gridsize=50, cmap='plasma', mincnt=1)
axes[0,0].set_xlabel('Gaia T_eff (K)')
axes[0,0].set_ylabel('Pan-STARRS Te_iz (K)')
axes[0,0].set_title('2D Histogram: Gaia vs Pan-STARRS Temperature')
axes[0,0].plot([3000, 8000], [3000, 8000], 'w--', alpha=0.7, label='1:1 line')
axes[0,0].legend()

# Density plot with contours
axes[0,1].hist2d(temp_data['teff_gspphot'], temp_data['Te_iz'],
                bins=100, cmap='viridis', density=True)
axes[0,1].set_xlabel('Gaia T_eff (K)')
axes[0,1].set_ylabel('Pan-STARRS Te_iz (K)')
axes[0,1].set_title('Density Plot with Contours')
axes[0,1].plot([3000, 8000], [3000, 8000], 'w--', alpha=0.7)

# Residuals vs Gaia T_eff
residuals = temp_data['Te_iz'] - temp_data['teff_gspphot']
axes[1,0].hexbin(temp_data['teff_gspphot'], residuals,
                gridsize=50, cmap='coolwarm', mincnt=1)
axes[1,0].set_xlabel('Gaia T_eff (K)')
axes[1,0].set_ylabel('Te_iz - T_eff (K)')
axes[1,0].set_title('Temperature Residuals')
axes[1,0].axhline(y=0, color='white', linestyle='--', alpha=0.7)

# Distribution of residuals
axes[1,1].hist(residuals, bins=100, alpha=0.7, density=True)
axes[1,1].set_xlabel('Te_iz - T_eff (K)')
axes[1,1].set_ylabel('Density')
axes[1,1].set_title('Residual Distribution')
axes[1,1].axvline(x=0, color='red', linestyle='--', alpha=0.7)
axes[1,1].axvline(x=np.median(residuals), color='orange', linestyle='-',
                    label=f'Median: {np.median(residuals):.0f} K')
axes[1,1].legend()

plt.tight_layout()
plt.show()

# Print statistics
print(f"\nTemperature comparison statistics:")
print(f"Gaia T_eff range: {temp_data['teff_gspphot'].min():.0f} - {temp_data['teff_gspphot'].max():.0f} K")
print(f"Pan-STARRS Te_iz range: {temp_data['Te_iz'].min():.0f} - {temp_data['Te_iz'].max():.0f} K")
print(f"Median residual (Te_iz - T_eff): {np.median(residuals):.0f} K")
print(f"MAD residual: {np.median(np.abs(residuals - np.median(residuals))):.0f} K")
print(f"RMS residual: {np.sqrt(np.mean(residuals**2)):.0f} K")
